In [0]:
# --------------------------------------------------------------
# Set environment variables for the pipeline run
# --------------------------------------------------------------
# These values are retrieved from the master pipeline task 
# (superstore_pipeline_master_run_id_init) using dbutils.jobs.taskValues.
# They ensure that the ETL, Mart, Features, KPI, and Dashboard code
# runs in the correct environment (dev/qa/prod) without hardcoding paths.
# SUPERSTORE_ENV, SUPERSTORE_PIPELINE_NAME, and SUPERSTORE_PIPELINE_VERSION
# are set for downstream modules to dynamically resolve catalog/schema.
# --------------------------------------------------------------
import os

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -------------------------------
# Environment Config
# -------------------------------
import sys
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_shared_utilities")
from superstore_platform_config import get_catalog, get_gold_schema, get_mart_schema

# Dynamically get environment-aware catalog and schemas
catalog = get_catalog()            # e.g., superstore_catalog
gold_schema = get_gold_schema()    # e.g., dev_gold
mart_schema = get_mart_schema() 

# -------------------------------
# Create mart Table (Environment-aware)
# -------------------------------
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{mart_schema}.mart_sales_daily AS
SELECT
  o.order_date,
  s.customer_id,
  s.product_id,
  c.region,
  c.segment,
  p.category,
  p.sub_category,
  o.ship_mode,

  SUM(s.sales) AS total_sales,
  SUM(s.profit) AS total_profit,
  SUM(s.quantity) AS total_quantity,
  AVG(s.discount) AS avg_discount,
  COUNT(DISTINCT o.order_id) AS order_count,

  CASE 
   WHEN SUM(s.sales) = 0 THEN NULL 
   ELSE ROUND(SUM(s.profit) / SUM(s.sales), 4) 
  END AS profit_margin

FROM {catalog}.{gold_schema}.facts_orders o
JOIN {catalog}.{gold_schema}.facts_sales s 
 ON o.order_id = s.order_id
JOIN {catalog}.{gold_schema}.dim_customers c 
 ON s.customer_id = c.customer_id AND c.is_current = true
JOIN {catalog}.{gold_schema}.dim_products p 
 ON s.product_id = p.product_id AND p.is_current = true

GROUP BY
  o.order_date,
  s.customer_id,
  s.product_id,
  c.region,
  c.segment,
  p.category,
  p.sub_category,
  o.ship_mode
;
""")
